# Paper tables

In [1]:
from pathlib import Path
import sys

working_dir = Path.cwd()
root = next(
    (path for path in (working_dir, *working_dir.parents) if (path / "inputs" / "prompts.json").is_file()),
    None,
)
if root is None:
    raise FileNotFoundError("Open this notebook from the final_project directory or one of its subdirectories.")
sys.path.insert(0, str(root / "code"))

In [2]:
from IPython.display import display
import pandas as pd

from utils.paper_data import DATASETS_DIR, MARKETS, load_paper_data

tables_dir = root / "outputs" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
listings, interiors, quality_models = load_paper_data(root)


def save_table(table, number, **options):
    display(table)
    path = tables_dir / f"table_{number}.tex"
    path.write_text(table.to_latex(float_format="{:.3f}".format, escape=True, na_rep="", **options), encoding="utf-8")


def coefficient_cell(model, term):
    if term not in model.params:
        return ""
    p = model.pvalues[term]
    stars = "***" if p < .01 else "**" if p < .05 else "*" if p < .10 else ""
    return f"{model.params[term]:.3f}{stars}"

## Methods and descriptives

Tables 1–4. The listing summaries use the current photo eligibility.
Image descriptives and the image-quality regression use the same finished-interior sample.

### Table 1. CLIP polarity calculation

In [3]:
steps = pd.DataFrame({"Calculation": [
    "Use the saved similarities for the positive, neutral, negative, and unfinished prompts.",
    "Multiply the similarities by the learned CLIP scale to obtain logits.",
    "Apply softmax across the four logits to obtain class probabilities.",
    "Flag interiors as unfinished when their unfinished probability is at least 0.90.",
    "For finished photos, condition on the three finished classes and subtract the negative probability from the positive probability.",
]}, index=pd.Index(["Step 1", "Step 2", "Step 3", "Step 4", "Step 5"], name="Step"))
save_table(steps, "01", column_format=r"lp{0.73\linewidth}")

,Calculation
Step,
Step 1,"Use the saved similarities for the positive, n..."
Step 2,Multiply the similarities by the learned CLIP ...
Step 3,Apply softmax across the four logits to obtain...
Step 4,Flag interiors as unfinished when their unfini...
Step 5,"For finished photos, condition on the three fi..."


### Table 2. Sample means

In [4]:
categorical = {
    "roomsCount_grouped": "Number of rooms",
    "bathrooms_cat": "Number of bathrooms",
    "ceilingHeightCapped": "Ceiling height (m)",
    "apartment": "Apartment",
    "toDemolish": "Demolition program",
    "floorNumberCapped": "Floor number",
    "material": "Construction material",
    "repairType": "State of repair",
}
numerical = {
    "totalArea": "Area (square metres)",
    "age": "Building age (years)",
    "distance_center": "Distance to center",
    "distance_metro": "Distance to metro",
    "distance_nice": "Distance to green area or water",
    "distance_ugly": "Distance to industrial zone",
}
means = {}
for market, title in MARKETS.items():
    data = listings.loc[listings["deal_type"].eq(market) & listings["link_num"].gt(0)]
    rows = {(label, "Mean"): data[column].mean() for column, label in numerical.items()}
    for column, label in categorical.items():
        shares = data[column].value_counts(normalize=True, sort=False)
        if column in ("apartment", "toDemolish"):
            shares = shares.loc[["yes"]]
        rows.update({(label, level): share for level, share in shares.items()})
    rows["Listings with usable photos", "N"] = f"{len(data):,}"
    means[title] = pd.Series(rows)

table = pd.DataFrame(means)
table.index.names = ["Variable", "Category"]
# Geographic distances are in units of 100 metres.
save_table(table, "02")

Long-term Rentals Secondary Sales
Variable                        Category                                  
Area (square metres)            Mean             54.935912        61.99837
Building age (years)            Mean             32.470896       29.467018
Distance to center              Mean            110.760428      108.982078
Distance to metro               Mean               7.01613        7.190764
Distance to green area or water Mean              7.066346        6.730014
Distance to industrial zone     Mean              5.580034        5.428716
Number of rooms                 1                 0.379348         0.28564
                                2                 0.404637        0.381343
                                3                 0.181895        0.260915
                                4+                 0.03412        0.072102
Number of bathrooms             1                 0.870247        0.789401
                                2+                0.129753        0.210599
Ceiling height (m)              2.4-2.7           0.175167        0.239042
                                2.7-3.0           0.154322         0.19011
                                3.0+              0.107106         0.25884
                                NR                0.563405        0.312008
Apartment                       yes               0.040127        0.068384
Demolition program              yes               0.051721        0.023256
Floor number                    1st                0.05088        0.055243
                                2-9               0.604794        0.575949
                                10-19             0.264672        0.264113
                                20-29             0.057248         0.06752
                                30+               0.022406        0.037175
Construction material           brick             0.164654        0.161235
                                block             0.079534        0.071496
                                panel             0.328648        0.303795
                                concrete          0.226708        0.356013
                                NR                0.200457        0.107461
State of repair                 Designer          0.227428        0.231002
                                Euro              0.382591        0.208438
                                Cosmetic          0.297531        0.299127
                                None              0.011173        0.186306
                                NR                0.081276        0.075128
Listings with usable photos     N                   16,647          11,567

### Table 3. Image quality descriptives

In [5]:
image_columns = {
    "ratio": "Aspect ratio",
    "resolution": "Resolution (million pixels)",
    "sharpness": "Sharpness (thousands)",
    "brightness": "Brightness",
    "brisque": "BRISQUE",
    "clip_polarity": "CLIP polarity",
}
finished = interiors.dropna(subset=["clip_polarity_adj"])
quality_stats = {}
for market, title in MARKETS.items():
    photos = finished.loc[finished["deal_type"].eq(market)]
    stats = photos[list(image_columns)].describe().loc[["mean", "std", "min", "50%", "max"]].T
    quality_stats[title] = stats.rename(index=image_columns, columns={"50%": "median"})
    print(f"{title}  N = {len(photos):,}")
save_table(pd.concat(quality_stats, axis=1), "03")

Long-term Rentals  N = 291,090
Secondary Sales  N = 192,533


Long-term Rentals                                \
                                         mean       std       min    median   
Aspect ratio                         1.018845  0.381633  0.278125  0.750000   
Resolution (million pixels)          0.820031  0.248266  0.043264  0.691200   
Sharpness (thousands)                0.579571  0.781607  0.001260  0.362179   
Brightness                           0.516190  0.088896  0.026987  0.511120   
BRISQUE                              0.265013  0.124209  0.000000  0.254673   
CLIP polarity                        0.252366  0.317555 -0.980199  0.231578   

                                       Secondary Sales                      \
                                   max            mean       std       min   
Aspect ratio                  5.000000        1.079700  0.386868  0.290625   
Resolution (million pixels)   1.637120        0.855065  0.251110  0.043264   
Sharpness (thousands)        10.000000        0.568733  0.737282  0.001191   
Brightness                    0.978570        0.520426  0.095643  0.052832   
BRISQUE                       1.000000        0.271449  0.127036  0.000000   
CLIP polarity                 0.979375        0.222716  0.379731 -0.991464   

                                                  
                               median        max  
Aspect ratio                 1.000000   4.785714  
Resolution (million pixels)  0.692160   1.637120  
Sharpness (thousands)        0.370290  10.000000  
Brightness                   0.514985   0.986734  
BRISQUE                      0.259668   1.000000  
CLIP polarity                0.207095   0.977867

### Table 4. Regression of CLIP polarity on image quality

In [6]:
results = {}
for market, model in quality_models.items():
    rows = {}
    for term, label in image_columns.items():
        if term in model.params:
            rows[label] = coefficient_cell(model, term)
            rows[f"{label} (clustered SE)"] = f"({model.bse[term]:.3f})"
    rows["Observations"] = f"{model.nobs:,.0f}"
    rows["R²"] = f"{model.rsquared:.3f}"
    rows["F-statistic"] = f"{model.fvalue:.1f}"
    results[MARKETS[market]] = rows
save_table(pd.DataFrame(results), "04")

,Long-term Rentals,Secondary Sales
Aspect ratio,0.045***,0.098***
Aspect ratio (clustered SE),(0.006),(0.009)
Resolution (million pixels),0.059***,0.015
Resolution (million pixels) (clustered SE),(0.009),(0.013)
Sharpness (thousands),0.021***,0.048***
Sharpness (thousands) (clustered SE),(0.002),(0.003)
Brightness,0.620***,0.791***
Brightness (clustered SE),(0.017),(0.026)
BRISQUE,0.179***,0.342***
BRISQUE (clustered SE),(0.010),(0.014)


## Price models and time on market

Tables 5–7 and the model-based numbers use listings with adjusted interior Q
and the required listing data. The price models share a train/test split
within each market. Model 1 uses listing controls. Model 2 adds adjusted
interior Q.

### Fit OLS and histogram gradient boosting

In [7]:
from utils.paper_data import load_llama_scores
from utils.paper_models import (
    PHOTO_FEATURES, FEATURE_LABELS,
    boost_features, fit_price_models, model_performance, ols_features,
)
from utils.paper_survival import duration_sample

fits = {market: fit_price_models(listings, market) for market in MARKETS}
durations = {market: duration_sample(listings, market) for market in MARKETS}

### Tables 5 and 6. Hedonic regression and gradient boosting

In [8]:
ols_columns, hgb_columns = {}, {}
test_performance = {}
for market, (train, test, models) in fits.items():
    for number, model in models.items():
        key = (MARKETS[market], f"({number})")
        ols = model["ols"]
        ols_rows = {}
        for term in PHOTO_FEATURES:
            label = FEATURE_LABELS[term]
            ols_rows[label] = coefficient_cell(ols, term)
            ols_rows[f"{label} (HC3 SE)"] = f"({ols.bse[term]:.3f})" if term in ols.params else ""
        hgb_rows = {}
        for split, data in [("Train", train), ("Test", test)]:
            actual = data["log_price"]
            ols_prediction = ols.predict(ols_features(model["design_info"], data))
            hgb_prediction = model["hgb"].predict(boost_features(data, model["features"]).to_numpy())
            ols_stats = model_performance(actual, ols_prediction, int(ols.df_model))
            hgb_stats = model_performance(actual, hgb_prediction, len(model["features"]))
            if split == "Test":
                test_performance[market, number] = {"ols": ols_stats, "hgb": hgb_stats}
            for label, result in ols_stats.items():
                ols_rows[f"{split} {label}"] = f"{result:,.0f}" if label == "N" else f"{result:.3f}"
            for label, result in hgb_stats.items():
                hgb_rows[f"{split} {label}"] = f"{result:,.0f}" if label == "N" else f"{result:.3f}"
        for rows in (ols_rows, hgb_rows):
            rows["Original listing features"] = "Yes"
            rows["Interior Q-score"] = "Yes" if number == 2 else ""
        ols_columns[key] = ols_rows
        hgb_columns[key] = hgb_rows

save_table(pd.DataFrame(ols_columns), "05")
save_table(pd.DataFrame(hgb_columns), "06")

Long-term Rentals  ... Secondary Sales
                                                 (1)  ...             (2)
Adjusted interior Q-score                             ...        0.198***
Adjusted interior Q-score (HC3 SE)                    ...         (0.012)
Train N                                       13,090  ...           8,648
Train Adjusted R²                              0.874  ...           0.911
Train RMSE                                     0.179  ...           0.185
Test N                                         3,273  ...           2,162
Test Adjusted R²                               0.860  ...           0.910
Test RMSE                                      0.184  ...           0.176
Original listing features                        Yes  ...             Yes
Interior Q-score                                      ...             Yes

[10 rows x 4 columns]

Long-term Rentals         Secondary Sales       
                                        (1)     (2)             (1)    (2)
Train N                              13,090  13,090           8,648  8,648
Train Adjusted R²                     0.928   0.938           0.951  0.955
Train RMSE                            0.136   0.126           0.138  0.133
Test N                                3,273   3,273           2,162  2,162
Test Adjusted R²                      0.883   0.900           0.922  0.928
Test RMSE                             0.172   0.159           0.170  0.163
Original listing features               Yes     Yes             Yes    Yes
Interior Q-score                                Yes                    Yes

### Table 7. Time on the market

In [9]:
summaries = {}
for market, data in durations.items():
    summary = data.groupby("status")["dur"].agg(["count", "mean", "std", "min", "median", "max"])
    summary = summary.reindex([True, False]).rename(index={True: "Closed", False: "Censored"})
    summary["count"] = summary["count"].map(lambda n: f"{n:,.0f}")
    summaries[MARKETS[market]] = summary.round(1)
save_table(pd.concat(summaries, names=["Market", "Status"]), "07")

count   mean   std   min  median    max
Market            Status                                            
Long-term Rentals Closed    15,663   28.1  30.4   1.2    16.9  156.6
                  Censored     637  126.6  17.6  60.5   126.5  156.6
Secondary Sales   Closed     8,814   52.6  37.5   1.3    43.3  157.8
                  Censored   1,958  128.8  16.9  60.0   128.7  157.4

## Appendix

### Table A.1. LLaMA4 image-rating prompt

In [10]:
prompts = pd.Series({
    "System prompt": "You are a helpful assistant that can look at pictures of real estate and provide accurate descriptions and analysis. Answer the user's questions regarding the attached images as clearly and accurately as possible.",
    "User query": "Rate how luxurious and upscale this apartment looks based on the pictures. Give a score from 0 to 10 to each photo. Answer with a list of scores (e.g.: '[4,3,5,7,2,0,9]'). Omit any explanation or additional information.",
    "Image inputs": "Up to ten photo URLs, randomly selected when a listing has more than ten photos.",
    "Decoding": "Temperature 0",
}, name="Prompt or setting")
save_table(prompts.to_frame(), "A1", column_format=r"lp{0.73\linewidth}")

,Prompt or setting
System prompt,You are a helpful assistant that can look at p...
User query,Rate how luxurious and upscale this apartment ...
Image inputs,"Up to ten photo URLs, randomly selected when a..."
Decoding,Temperature 0


## Computed numbers cited in the text

In [11]:
llama_path = root / DATASETS_DIR / "llama4_scores.parquet"
llama = load_llama_scores(llama_path, listings) if llama_path.is_file() else None
numbers = []
for market in MARKETS:
    data = listings.loc[listings["deal_type"].eq(market)]
    photos = interiors.loc[interiors["deal_type"].eq(market)]
    usable = data["link_num"].gt(0)
    train, test, models = fits[market]
    duration = durations[market]
    metrics = {
        "listings": len(data),
        "first_listing_date": data["date0"].min().date().isoformat(),
        "last_listing_date": data["date0"].max().date().isoformat(),
        "last_observed_date": data["date1"].max().date().isoformat(),
        "mean_new_listings_per_day": data.groupby(data["date0"].dt.date).size().mean(),
        "owner_share": data["author_cat"].eq("Owner").mean(),
        "fraction_without_usable_photos": (~usable).mean(),
        "listings_with_usable_photos": usable.sum(),
        "usable_photos": data["link_num"].sum(),
        "interior_photos": len(photos),
        "landscape_share_of_interiors_with_dimensions": photos["ratio"].dropna().gt(1).mean(),
        "jpeg_interior_photos": photos["format"].eq("JPEG").sum(),
        "grayscale_interior_photos": photos["mode"].eq("L").sum(),
        "unfinished_interior_photos": photos["unfinished"].sum(),
        "listings_with_all_scored_interiors_unfinished": data["share_unfinished"].eq(1).sum(),
        "finished_interiors_with_quality_metrics": int(quality_models[market].nobs),
        "fraction_polarity_variance_unexplained": 1 - quality_models[market].rsquared,
        "listings_with_adjusted_interior_score": data["clip_polarity_adj"].count(),
        "listings_with_exterior_score": data["exterior_polarity"].count(),
        "listings_with_floor_plan_score": data["plan_polarity"].count(),
        "price_train_listings": len(train),
        "price_test_listings": len(test),
        "ols_interior_price_percent_change_per_point_one": 100 * 0.1 * models[2]["ols"].params["clip_polarity_adj"],
        "duration_listings": len(duration),
        "right_censored_share": duration["status"].eq(False).mean(),
    }
    if llama is not None:
        ratings = llama.loc[llama["deal_type"].eq(market) & llama["link_num"].gt(0)]
        comparable = ratings[["clip_polarity", "average_luxury_score"]].dropna()
        metrics["llama_scored_listings"] = ratings["average_luxury_score"].count()
        metrics["llama_scored_share"] = ratings["average_luxury_score"].notna().mean()
        metrics["clip_llama_comparable_listings"] = len(comparable)
        metrics["clip_llama_pearson_correlation"] = comparable["clip_polarity"].corr(comparable["average_luxury_score"])
    for kind in ("ols", "hgb"):
        baseline = test_performance[market, 1][kind]["Adjusted R²"]
        metrics[f"{kind}_test_adj_r2_gain_interior"] = test_performance[market, 2][kind]["Adjusted R²"] - baseline
    numbers.extend((label, market, value) for label, value in metrics.items())

numbers = pd.DataFrame(numbers, columns=["metric", "deal_type", "value"])
numbers.to_csv(tables_dir / "paper_numbers.csv", index=False)
display(numbers.pivot(index="metric", columns="deal_type", values="value"))

deal_type,long_rent,sale_secondary
metric,,
clip_llama_comparable_listings,14090,9680
clip_llama_pearson_correlation,0.77842,0.723806
duration_listings,16300,10772
finished_interiors_with_quality_metrics,291090,192533
first_listing_date,2025-07-01,2025-07-01
fraction_polarity_variance_unexplained,0.946738,0.910056
fraction_without_usable_photos,0.034901,0.017414
grayscale_interior_photos,0,0
hgb_test_adj_r2_gain_interior,0.016635,0.006067
